# OVITO DXA：Notebook 默认多进程分析

逐个运行单元格：检查环境 → 生成并确认示例清单 → 只读预检 → `%%writefile DXA_Multiprocessing.py` → `%run DXA_Multiprocessing.py`。

**默认不运行单进程基线或性能比较。** 需要比较时，文末提供可选的小样本命令。这里只使用公开生成的 BCC/FCC 完整晶体；实际研究数据必须先确认分组、晶格、原子选择、单位、边界和输出档次。

In [ ]:
from pathlib import Path  # 定位真实的仓库和本次示例目录。
import sys  # 读取当前 Jupyter 内核使用的解释器。
import os  # 给预检子进程提供明确的 UTF-8 输出编码。
import json  # 读取已经生成的任务清单和实际运行记录。
import shutil  # 将可导入的工作模块复制到本次示例目录。
import subprocess  # 只用于环境检查、生成输入和只读预检，DXA 主流程使用 %run。
from datetime import datetime, timezone  # 为每次示例创建不重复的目录名。
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tests/make_examples.py").is_file()), None)  # 从当前目录向上定位仓库。
if root is None:  # 避免在未知目录猜测研究文件位置。
    raise RuntimeError("请从仓库或它的 examples 目录打开此 Notebook。")  # 只指出当前需要解决的问题。
python = sys.executable  # %run 和它启动的进程池均使用此内核解释器。
child_env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")  # 子进程输出中文时采用统一编码。
probe = subprocess.run([python, "-c", "import ovito, psutil; print(ovito.version_string)"], check=True, capture_output=True, text=True, encoding="utf-8", env=child_env)  # 检查已有依赖，不修改环境。
print("解释器：", python, "\nOVITO：", probe.stdout.strip())  # 展示实际运行环境供检查。

## 生成并检查示例清单

每次创建新目录，不覆盖旧结果。本示例的三帧 BCC 轨迹和一份 FCC 构型为两组独立数据；生成器明确指定其科学参数，不能直接把这些设置套给研究数据。清单使用 `workers="auto"`，按当前可用 CPU、内存预算和实际帧块数选择进程数，每进程默认一个 OVITO 线程。

In [ ]:
example_dir = root / "examples" / ("notebook_generated_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f"))  # 使用本次独立目录。
subprocess.run([python, str(root / "tests/make_examples.py"), "--output", str(example_dir)], check=True, env=child_env)  # 生成真实合成数据与完整任务清单。
config = example_dir / "task.json"  # 指向刚刚实际创建的清单，不使用占位路径。
settings = json.loads(config.read_text(encoding="utf-8"))  # 读取待分析范围和设置。
print(json.dumps(settings, ensure_ascii=False, indent=2))  # 在执行 DXA 前查看两组数据与资源参数。

## 只读预检

预检读取粒子和帧信息，不计算 DXA，也不是单进程计时对照。对于关系不明确的文件批次，检查器只提出候选分组，仍须向使用者确认。

In [ ]:
inspector = root / "skills/ovito-dxa/scripts/inspect_inputs.py"  # 定位已有只读检查器。
inspection = subprocess.run([python, str(inspector), "--paths", str(example_dir / "bcc.dump"), str(example_dir / "fcc.data"), "--all-frames"], check=True, capture_output=True, text=True, encoding="utf-8", env=child_env)  # 检查两组示例输入是否可读。
print(inspection.stdout)  # 查看实际帧数、属性及候选确认项。

## 准备可导入的运行模块

Windows 的 `spawn` 子进程需要导入 worker 函数。因此，`DXA_Multiprocessing.py` 是入口，实际 worker 保存在旁边 `dxa_runtime/run_dxa.py` 中。不能把这些 worker 临时定义在 Notebook 的 `__main__`。

`%run` 使用当前内核的解释器；请先选择已有 OVITO 环境的内核。

In [ ]:
runtime_dir = example_dir / "dxa_runtime"  # 将运行器与本次任务放在同一明确目录下。
runtime_dir.mkdir(exist_ok=False)  # 拒绝覆盖已有运行模块。
for filename in ("run_dxa.py", "inspect_inputs.py"):  # 运行器引用同目录检查器，二者一起复制。
    shutil.copy2(root / "skills/ovito-dxa/scripts" / filename, runtime_dir / filename)  # 复制已验证的模块而不改动原文件。
# 切换到实际创建的目录，后续魔法命令直接使用真实相对路径。
%cd -q {example_dir}

## 第一步：写入脚本

下面的 `%%writefile` 只保存代码，不启动计算。入口把已确认的 `task.json` 交给运行器；不传 `--mode` 时默认直接正式分析。若更换了运行器版本，需要重启内核后重新导入，避免复用旧模块。

In [ ]:
%%writefile DXA_Multiprocessing.py
__spec__ = None  # 将 IPython 重复执行的入口明确标为脚本，供 Windows spawn 读取。
from pathlib import Path  # 以脚本自身位置定位清单和运行模块。
import sys  # 将运行模块目录加入子进程同样可用的导入路径。
import multiprocessing as mp  # 提供 Windows 进程入口保护支持。
work_dir = Path(__file__).resolve().parent  # 不依赖 Notebook 后续是否切换工作目录。
runtime_dir = work_dir / "dxa_runtime"  # 使用前一单元已经复制的模块目录。
if not (runtime_dir / "run_dxa.py").is_file():  # 缺少运行模块时不能假装启动成功。
    raise FileNotFoundError("缺少 dxa_runtime/run_dxa.py，请先完成运行模块准备。")  # 明确当前缺少的文件。
sys.path.insert(0, str(runtime_dir))  # 让 spawn 子进程能按模块名导入 worker。
import run_dxa  # 导入函数定义；此操作不会在 Notebook 主进程导入 OVITO。
if Path(run_dxa.__file__).resolve() != (runtime_dir / "run_dxa.py").resolve():  # 防止内核缓存了其他目录的同名旧模块。
    raise RuntimeError("当前内核已加载另一份 run_dxa；请重启内核后从头执行。")  # 避免静默使用旧实现。
if __name__ == "__main__":  # 子进程导入时不重复启动主流程。
    mp.freeze_support()  # 保留 Windows 独立脚本的标准入口保护。
    run_dxa.main(["--config", str(work_dir / "task.json"), *sys.argv[1:]])  # 默认直接多进程；仅显式传入参数时切换模式。

## 第二步：直接多进程运行

当前示例只有四个合成帧，直接运行全部已确认帧，不先跑单进程。资源充足且每帧一块时最多开四个进程；换一台机器或换一份数据，实际数量可以不同。省略资源设置时也按 `workers="auto"` 自动选择进程数，每进程默认一个线程。实际进程数取 CPU 预算、内存预算和待算帧块数等上限中的最小值；默认保留约 10% 的 CPU 预算和 25% 的可用内存。内存估算复用输入预检中的粒子数量，不额外做单进程 DXA 试跑，也不保证实测最快。

只有一个可并行帧块，或 CPU／内存预算只允许一个进程时，会收窄到一个 worker，这不是额外的单进程对照。`%run` 自身不是并行开关；并行来自运行器内部的 `spawn` 进程池。

In [ ]:
# 在当前 OVITO 内核中执行刚写入的脚本；默认模式是 run。
%run DXA_Multiprocessing.py

In [ ]:
result_dir = example_dir / "results/task/run"  # 对应当前清单里真实的输出目录。
execution = json.loads((result_dir / "execution.json").read_text(encoding="utf-8"))  # 读取实际进程数、耗时及完成数量。
print(json.dumps(execution, ensure_ascii=False, indent=2))  # 检查执行配置和结果是否完整。
print("结果目录：", result_dir)  # 提供本次输出的真实位置。

## 需要时续算

重复 `run` 会拒绝覆盖。`resume` 核对输入、参数、版本及输出摘要，只补算失败或缺失帧；已全部完成时不再执行 DXA。

In [ ]:
# 演示完整结果可复用；这一步不会把全部成功帧重新计算一遍。
%run DXA_Multiprocessing.py --mode resume

## 可选：少量代表帧性能比较

**默认流程不执行本节命令；仅在使用者要求性能对比时手动执行。** 在已生成脚本的目录，可用 `%run DXA_Multiprocessing.py --mode benchmark` 比较少量代表帧。

每组先选首、中、末候选，再按 `resources.benchmark_max_frames` 全局限制，默认总共最多 6 帧；需要更少时先在清单中明确修改上限。仅这些样本会运行单进程及并行候选，不再额外运行 `baseline`，也不运行全量串行对照。未抽中的工况会记入报告，不能由样本宣称整批数据一定更快。

报告位于当前示例的 `results/task/benchmark/report.html`，包含耗时、采样峰值内存和数值一致性。小数据可能因启动成本而没有加速，保留实测结果，不自动替换正式运行配置。

## 已融合的优化方法

每个 worker 自己建立并复用 OVITO 流水线；DXA 只添加一次，同一帧指标共享一次计算；各帧只回传小型统计，主进程集中写汇总表。几何档的 CA 文件由 worker 独立落盘。使用连续帧块可减少调度和重复读取，应按实际负载设置。

`bcc.dump` 与三个 `bcc.*.dump` 是同一轨迹的两种保存方式，不是四组独立工况。真实输入不能仅凭文件名推断这种关系。